In [4]:
import sys
sys.path.append("/Users/bubble/Desktop/Project/T_sensor/T_sensor")
from Tools.geo_trans import round_corner

import gdsfactory as gf
# TODO
# 1. Change one side of the cell_temp to be comparation with the width
# 2. Different with the width and same width with gap

cell_temp = gf.Component()

In [6]:
# cell
L = [500, 500, 1000, 1000]
w = [0.5, 1, 1.5, 2]
k = len(L) * len(w)

side =[None] * len(L)

for i in range(len(w)):
    side[i] = gf.Component()
block = gf.Component()
origin = (5000 / (len(w)+1), 0)

for i in range(len(L)):
    for j in range(len(w)):
        offset = 10
        # ebeam layer
        ebl_beam = gf.components.rectangle(size=(w[j], L[i]), layer=(5, 0))
        (side[i] << ebl_beam).move((origin[0]+5000/(len(w)+1)*j, origin[1]))
        ebl_frame = gf.components.rectangle(size=(w[j]+2*offset, L[i]+offset), layer=(6, 0))
        (side[i] << ebl_frame).move((origin[0]+5000/(len(w)+1)*j - offset, origin[1]))
        T_corner = round_corner(4*w[j], w[j], 2*w[j], rotation=90, layer=(5, 0))
        corner_ref1 = side[i] << T_corner
        corner_ref1.move((origin[0]+5000/(len(w)+1)*j+ w[j]/2, origin[1]))
        # optical beam
        opt_beam = gf.components.rectangle(size=(w[j]+offset, L[i]+offset/2), layer=(8, 0))
        (side[i] << opt_beam).move((origin[0]+5000/(len(w)+1)*j - offset/2, origin[1]))
        # length mark
        T = gf.components.text(f"L={L[i]} w={w[j]}", size=50, layer=(1, 0))
        T_ref = side[i] << T
        T_ref.move((origin[0]+5000/(len(w)+1)*j-200, origin[1]-150))
            

for i in range(4):
    side_ref = block << side[i]
    if i == 0:
        pass
    elif i == 1:
        side_ref.dmirror_y(5000/2).dmirror_x(5000/2)
    elif i == 2:
        side_ref.drotate(angle=-90, center=(5000/2, 5000/2))
    else:
        side_ref.drotate(angle=90, center=(5000/2, 5000/2))
opt_frame = gf.components.rectangle(size=(5000, 5000), layer=(9, 0))
block << opt_frame


block.show()

In [17]:
# backside etching 5mm * 5mm (5743.44um)
backside = gf.Component()
size = 5743.44
diff = (size - 5000) / 2
for i in range(4):
    backside_temp = gf.components.rectangle(size=(size, size), layer=(3, 0))
    backside_ref = backside << backside_temp
    if i == 0:
        backside_ref.move((-diff, -diff))
    elif i == 1:
        backside_ref.move((10000-diff, -diff))
    elif i == 2:
        backside_ref.move((-diff, 10000-diff))
    else:
        backside_ref.move((10000-diff, 10000-diff))
# backside.show()

In [18]:
# structure for each die
# structure for a block
fblock = gf.Component()
for i in range(4):
    if i == 0:
        block_ref = fblock << block
    elif i == 1:
        block_ref = fblock << block
        block_ref.move((10000, 0))
    elif i == 2:
        block_ref = fblock << block
        block_ref.move((10000, 10000))
    else:
        block_ref = fblock << block
        block_ref.move((0, 10000))

In [ ]:


# order
order = gf.Component()
text_array = []
for i in range(4):
    text_array.append(gf.Component())

for i in range(4):
    T = gf.components.text(f"UTB {i+1}", size=20, layer=(1, 0))
    for j in range(4):
        order_ref = text_array[i] << T
        if j == 0:
            order_ref.move((-65, 0))
        elif j == 1:
            order_ref.move((-65, 5000))
        elif j == 2:
            order_ref.move((5020, 0))
        else:
            order_ref.move((5020, 5000))
    text_array_ref = order << text_array[i]
    if i == 0:
        pass
    elif i == 1:
        text_array_ref.move((10000, 0))
    elif i == 2:
        text_array_ref.move((0, 10000))
    else:
        text_array_ref.move((10000, 10000))





In [20]:
# backside frame 5mm * 5mm 
backframe = gf.Component()
size = 5000
for i in range(4):
    backframe_temp = gf.components.rectangle(size=(size, size), layer=(3, 0))
    backframe_ref = backframe << backframe_temp
    if i == 0:
        backframe_ref.move((0, 0))
    elif i == 1:
        backframe_ref.move((10000, 0))
    elif i == 2:
        backframe_ref.move((0, 10000))
    else:
        backframe_ref.move((10000, 10000))
# backside.show()

In [ ]:
# boolean operation
# do backside etching - frame
ebl = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(6, 0), layer2=(5, 0), layer=(5, 0))
cell_temp << ebl

opt = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(9, 0), layer2=(8, 0), layer=(1, 0))
cell_temp << opt

cell_temp << backside
# add order
marker = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(1, 0), layer2=(10, 0), layer=(1, 0))
cell_temp << marker
cell_temp << order
# frame
frame1 = gf.components.rectangle(size=(15000, 15000), layer=(20, 0))
frame2 = gf.components.rectangle(size=(20000, 20000), layer=(21, 0))
frame2_ref = cell_temp << frame2
frame2_ref.move((-2500, -2500))
cell_temp << frame1

cell_ultrathin_beam = gf.Component()
cell_ref = cell_ultrathin_beam << cell_temp
cell_ref.move((2500, -7500))
cell_ultrathin_beam.show()
# cell_ultrathin_beam.write_gds("mesh.gds")
# cell_ultrathin_beam.plot()